# Preparation

In [2]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.1 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random

# Generating Synthetic Data

In [4]:
# Initialize Faker and establish seeds for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print("Starting synthetic data generation for 'The Azure Stay'...")

# ==========================================
# 1. GENERATE DIMENSION TABLES
# ==========================================

# --- 1. dim_calendar ---
start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 12, 31)
calendar_dates = pd.date_range(start=start_date, end=end_date)
dim_calendar = pd.DataFrame({
    'date_key': calendar_dates.strftime('%Y-%m-%d'),
    'day_name': calendar_dates.day_name(),
    'is_weekend': calendar_dates.dayofweek.isin([5, 6]),
    'is_holiday': np.random.choice([True, False], size=len(calendar_dates), p=[0.03, 0.97]), # ~3% holidays
    'season': np.where(calendar_dates.month.isin([6, 7, 8, 12]), 'High Season',
              np.where(calendar_dates.month.isin([4, 5, 9, 10]), 'Shoulder Season', 'Low Season')),
    'event_flag': np.random.choice([True, False], size=len(calendar_dates), p=[0.02, 0.98]) # ~2% event days
})

# --- 2. dim_roomtype ---
dim_roomtype = pd.DataFrame({
    'room_type_id': ['RT_STD', 'RT_DLX', 'RT_SUI', 'RT_OCN'],
    'room_type_name': ['Standard Queen', 'Deluxe King', 'Suite', 'Ocean View'],
    'base_rate': [2500, 3800, 7500, 5000], # Thai Baht
    'capacity_count': [80, 40, 10, 20],
    'max_capacity': [2, 3, 4, 3]
})

# --- 3. dim_rate_codes ---
dim_rate_codes = pd.DataFrame({
    'rate_code_id': ['RC_RACK', 'RC_CORP', 'RC_PROMO', 'RC_NONREF'],
    'rate_name': ['Rack Rate', 'Corporate Negotiated', 'Seasonal Promo', 'Non-Refundable Discount'],
    'multiplier': [1.0, 0.80, 0.85, 0.75],
    'is_commissionable': [True, False, True, True]
})

# --- 4. dim_channels ---
dim_channels = pd.DataFrame({
    'channel_id': ['CH_DIR', 'CH_EXP', 'CH_BKG', 'CH_CORP'],
    'channel_name': ['Direct Website', 'Expedia', 'Booking.com', 'Corporate Agent'],
    'channel_type': ['Direct', 'OTA', 'OTA', 'Wholesale'],
    'commission_model': ['None', 'Percentage', 'Percentage', 'Net Rate'],
    'commission_rate': [0.0, 0.18, 0.15, 0.0], # 18% Expedia, 15% Booking
    'contract_owner': ['Internal', 'Alice Smith', 'Bob Jones', 'Charlie Davis']
})

# --- 5. dim_policies ---
dim_policies = pd.DataFrame({
    'policy_id': ['POL_FLEX', 'POL_48H', 'POL_NONREF'],
    'policy_name': ['Fully Flexible (24h)', '48 Hour Cancellation', 'Strict Non-Refundable'],
    'cancellation_deadline_hours': [24, 48, 8760], # 8760h = 1 year (effectively no cancellation)
    'penalty_type': ['First Night', 'First Night', 'Full Stay'],
    'is_refundable': [True, True, False]
})

# --- 6. dim_reason_cancellations ---
dim_reason_cancellations = pd.DataFrame({
    'cancellation_reason_code': ['RSN_01', 'RSN_02', 'RSN_03', 'RSN_04', 'RSN_05', 'RSN_06'],
    'reasons': ['Found Cheaper Price', 'Change of Plans', 'Weather/Flight Issue', 'Medical/Emergency', 'Work Conflict', 'Unknown']
})

# --- 7. dim_guests (Initial generation, metrics calculated later) ---
total_unique_guests = 6000 # Pool to draw 10k bookings from
guest_ids = [f'G_{str(i).zfill(5)}' for i in range(1, total_unique_guests + 1)]
countries = ['Thailand', 'USA', 'UK', 'China', 'Japan', 'Australia', 'Germany']

# Designate Serial Cancellers (Pattern 2: 6% of pool)
num_serial = int(total_unique_guests * 0.06)
serial_canceller_ids = set(guest_ids[:num_serial])

dim_guests = pd.DataFrame({
    'guest_id': guest_ids,
    'country_of_origin': np.random.choice(countries, size=total_unique_guests, p=[0.4, 0.1, 0.1, 0.15, 0.1, 0.05, 0.1]),
    'total_bookings_lifetime': 0, # Placeholder, updated after fact generation
    'total_cancellations_lifetime': 0,
    'cancellation_ratio': 0.0
})

# --- 8. dim_room_inventory ---
total_hotel_capacity = 150
dim_room_inventory = pd.DataFrame({
    'date': calendar_dates.strftime('%Y-%m-%d'),
    'total_capacity': total_hotel_capacity,
    'rooms_out_of_order': np.random.randint(0, 4, size=len(calendar_dates))
})
dim_room_inventory['rooms_available_for_sale'] = dim_room_inventory['total_capacity'] - dim_room_inventory['rooms_out_of_order']

# --- 9. fact_marketing_spend ---
spend_dates = pd.date_range(start=start_date, end=end_date, freq='W') # Weekly spend
fact_marketing_spend = pd.DataFrame({
    'spend_id': [f'MS_{str(i).zfill(4)}' for i in range(1, len(spend_dates) + 1)],
    'spend_date': spend_dates.strftime('%Y-%m-%d'),
    'channel_id': 'CH_DIR',
    'platform': np.random.choice(['Google Ads', 'Facebook Ads'], size=len(spend_dates)),
    'cost_amount': np.random.uniform(5000, 25000, size=len(spend_dates)).round(2),
    'clicks': np.random.randint(500, 3000, size=len(spend_dates))
})

# ==========================================
# 2. GENERATE FACT TABLE (fact_bookings)
# ==========================================
print("Generating 10,000 bookings with embedded insights...")

bookings = []
NUM_BOOKINGS = 10000

for i in range(NUM_BOOKINGS):
    booking_id = f'BKG_{str(i+1).zfill(6)}'

    # 1. Base Assignment
    guest_id = random.choice(guest_ids)
    is_serial_canceller = guest_id in serial_canceller_ids
    segment_id = random.choices(['Business', 'Leisure', 'Group'], weights=[0.25, 0.65, 0.10])[0]

    # 2. Lead Time & Check-in Date (Pattern 4: Business has short lead time)
    if segment_id == 'Business':
        lead_time = random.randint(0, 3)
        rate_code = 'RC_CORP'
        channel = random.choices(['CH_CORP', 'CH_DIR'], weights=[0.8, 0.2])[0]
        policy = 'POL_FLEX'
    else:
        lead_time = int(np.random.gamma(shape=2.0, scale=30.0)) # Skewed right, mean ~60 days
        rate_code = random.choices(dim_rate_codes['rate_code_id'].tolist(), weights=[0.4, 0.05, 0.35, 0.2])[0]
        channel = random.choices(dim_channels['channel_id'].tolist(), weights=[0.3, 0.3, 0.3, 0.1])[0]
        policy = random.choices(dim_policies['policy_id'].tolist(), weights=[0.6, 0.2, 0.2])[0]

    # Pattern 2 override: Serial Cancellers prefer Suites & Flex
    if is_serial_canceller:
        room_type = 'RT_SUI'
        policy = 'POL_FLEX'
    else:
        room_type = random.choices(dim_roomtype['room_type_id'].tolist(), weights=[0.6, 0.2, 0.1, 0.1])[0]

    # Calculate dates
    check_in_dt = start_date + timedelta(days=random.randint(30, 700))
    booking_dt = check_in_dt - timedelta(days=lead_time)

    # Length of Stay
    los = random.randint(1, 5) if segment_id == 'Business' else random.randint(1, 14)
    check_out_dt = check_in_dt + timedelta(days=los)

    # Deposit Logic
    deposit_paid = 0.0
    if policy == 'POL_NONREF':
        deposit_paid = 1.0 # Boolean proxy for full amount
    elif channel == 'CH_DIR' and random.random() > 0.4:
        deposit_paid = 1.0

    # 3. Status Determination (Embedding Patterns)
    # Base Probabilities
    prob_cancel = 0.15
    prob_noshow = 0.02

    is_weekend_arrival = check_in_dt.weekday() in [4, 5] # Friday or Saturday

    # PATTERN 1: The OTA Trap (OTA + Long Lead Time + Flex = High Cancel)
    if channel in ['CH_EXP', 'CH_BKG'] and lead_time > 60 and policy == 'POL_FLEX':
        prob_cancel = 0.50 # Drastically higher

    # PATTERN 2: Serial Cancellers (>80% cancel rate)
    if is_serial_canceller:
        prob_cancel = 0.85

    # PATTERN 4: Business Negotiated (Extremely low cancel rate)
    if segment_id == 'Business' and rate_code == 'RC_CORP':
        prob_cancel = 0.04

    # PATTERN 3: Weekend No-Show (Direct + Weekend + No Deposit)
    if channel == 'CH_DIR' and is_weekend_arrival and deposit_paid == 0.0:
        prob_noshow = 0.25 # Spike in no-shows

    # Determine final status
    rand_val = random.random()
    if rand_val < prob_cancel:
        status = 'Cancelled'
    elif rand_val < (prob_cancel + prob_noshow):
        status = 'No-Show'
    else:
        status = 'Checked-In'

    # Strict Non-Ref override (Less likely to cancel if they lose all money)
    if policy == 'POL_NONREF' and not is_serial_canceller:
        if random.random() < 0.8: # 80% forced to Check-in
            status = 'Checked-In'

    # 4. Cancellation Date & Reason Logic
    cancellation_date = pd.NA
    cancel_reason = pd.NA

    if status == 'Cancelled':
        cancel_reason = random.choice(dim_reason_cancellations['cancellation_reason_code'].tolist())

        deadline_hours = dim_policies[dim_policies['policy_id'] == policy]['cancellation_deadline_hours'].values[0]
        deadline_dt = check_in_dt - timedelta(hours=int(deadline_hours))

        if is_serial_canceller:
            # Pattern 2: Cancel exactly 24-48h before deadline
            cancellation_date = deadline_dt - timedelta(hours=random.randint(24, 48))
        else:
            # Normal cancel: Random time between booking and arrival
            delta_seconds = int((check_in_dt - booking_dt).total_seconds())
            if delta_seconds > 0:
                random_seconds = random.randint(0, delta_seconds)
                cancellation_date = booking_dt + timedelta(seconds=random_seconds)
            else:
                cancellation_date = booking_dt

        # Format date if not NA
        if pd.notna(cancellation_date):
            cancellation_date = cancellation_date.strftime('%Y-%m-%d')

        # Ensure 'Found Cheaper Price' is common for OTA Trap
        if channel in ['CH_EXP', 'CH_BKG'] and lead_time > 60:
            cancel_reason = 'RSN_01'

    # 5. Financial Calculations
    base_rate = dim_roomtype[dim_roomtype['room_type_id'] == room_type]['base_rate'].values[0]
    multiplier = dim_rate_codes[dim_rate_codes['rate_code_id'] == rate_code]['multiplier'].values[0]
    comm_rate = dim_channels[dim_channels['channel_id'] == channel]['commission_rate'].values[0]

    number_of_rooms = 1 # Simplified for this dataset

    gross_revenue = base_rate * multiplier * number_of_rooms * los

    penalty_charged = 0.0
    gross_room_revenue = 0.0
    total_room_revenue = 0.0

    if status == 'Checked-In':
        gross_room_revenue = gross_revenue
        total_room_revenue = gross_revenue
    elif status in ['Cancelled', 'No-Show']:
        # Penalty Logic based on policy
        if policy == 'POL_NONREF':
            penalty_charged = gross_revenue
        elif policy in ['POL_FLEX', 'POL_48H']:
            # Check if cancelled late
            if status == 'Cancelled' and pd.notna(cancellation_date):
                cancel_dt = datetime.strptime(cancellation_date, '%Y-%m-%d')
                deadline_hours = dim_policies[dim_policies['policy_id'] == policy]['cancellation_deadline_hours'].values[0]
                if (check_in_dt - cancel_dt).total_seconds() / 3600 < deadline_hours:
                    penalty_charged = base_rate * multiplier # 1st night fee
            if status == 'No-Show':
                penalty_charged = base_rate * multiplier # 1st night fee

        gross_room_revenue = penalty_charged
        total_room_revenue = penalty_charged

    commission_amount = gross_room_revenue * comm_rate
    net_room_revenue = gross_room_revenue - commission_amount

    # Append row
    bookings.append({
        'booking_id': booking_id,
        'guest_id': guest_id,
        'booking_date': booking_dt.strftime('%Y-%m-%d'),
        'check_in_date': check_in_dt.strftime('%Y-%m-%d'),
        'check_out_date': check_out_dt.strftime('%Y-%m-%d'),
        'room_type_id': room_type,
        'rate_code_id': rate_code,
        'channel_id': channel,
        'segment_id': segment_id,
        'policy_id': policy,
        'status': status,
        'total_room_revenue': round(total_room_revenue, 2),
        'number_of_rooms': number_of_rooms,
        'adults_count': random.randint(1, 2),
        'children_count': random.randint(0, 2),
        'cancellation_date': cancellation_date,
        'cancellation_reason_code': cancel_reason,
        'gross_revenue': round(gross_revenue, 2),
        'gross_room_revenue': round(gross_room_revenue, 2),
        'commission_amount': round(commission_amount, 2),
        'net_room_revenue': round(net_room_revenue, 2),
        'penalty_charged': round(penalty_charged, 2),
        'deposit_paid': deposit_paid
    })

fact_bookings = pd.DataFrame(bookings)

# ก่อนจะ Export ให้ทำการเรียงลำดับวันที่จอง และรัน ID ใหม่ให้สมเหตุสมผล
fact_bookings = fact_bookings.sort_values(by=['booking_date', 'check_in_date']).reset_index(drop=True)
fact_bookings['booking_id'] = ['BKG_' + str(i+1).zfill(6) for i in range(len(fact_bookings))]

# ==========================================
# 3. UPDATE DIM_GUESTS METRICS
# ==========================================
print("Updating Guest Dimensions based on Fact history...")

guest_stats = fact_bookings.groupby('guest_id').agg(
    total_bookings_lifetime=('booking_id', 'count'),
    total_cancellations_lifetime=('status', lambda x: (x == 'Cancelled').sum())
).reset_index()

guest_stats['cancellation_ratio'] = (guest_stats['total_cancellations_lifetime'] / guest_stats['total_bookings_lifetime']).round(2)

# Merge back to dim_guests
dim_guests = dim_guests.drop(columns=['total_bookings_lifetime', 'total_cancellations_lifetime', 'cancellation_ratio'])
dim_guests = dim_guests.merge(guest_stats, on='guest_id', how='left')

# Fill NaNs for guests who randomly didn't get any bookings in this 10k sample
dim_guests.fillna({'total_bookings_lifetime': 0, 'total_cancellations_lifetime': 0, 'cancellation_ratio': 0.0}, inplace=True)

# ==========================================
# 4. EXPORT TO CSV
# ==========================================
print("Exporting 10 DataFrames to CSV...")

dim_calendar.to_csv('dim_calendar.csv', index=False)
dim_roomtype.to_csv('dim_roomtype.csv', index=False)
dim_rate_codes.to_csv('dim_rate_codes.csv', index=False)
dim_channels.to_csv('dim_channels.csv', index=False)
dim_policies.to_csv('dim_policies.csv', index=False)
dim_reason_cancellations.to_csv('dim_reason_cancellations.csv', index=False)
dim_guests.to_csv('dim_guests.csv', index=False)
dim_room_inventory.to_csv('dim_room_inventory.csv', index=False)
fact_marketing_spend.to_csv('fact_marketing_spend.csv', index=False)

# Ensuring empty fields are handled cleanly without writing "NaN" string
fact_bookings.to_csv('fact_bookings.csv', index=False, na_rep='')

print("Success! 10 interconnected CSV files have been created. Fact table contains exactly", len(fact_bookings), "rows.")

Starting synthetic data generation for 'The Azure Stay'...
Generating 10,000 bookings with embedded insights...
Updating Guest Dimensions based on Fact history...
Exporting 10 DataFrames to CSV...
Success! 10 interconnected CSV files have been created. Fact table contains exactly 10000 rows.
